In [16]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import mat73
import mne
import eelbrain
from eelbrain import Dataset, Var, Factor, NDVar, UTS, Scalar, Sensor

## Settings

In [23]:
MAINDIR = Path(r"C:/projects/emo_EEG")
DATA_DIR = MAINDIR / "data" / "processed"
STIMULI_DIR = MAINDIR / "stimuli"
ORDER_DIR = STIMULI_DIR / "orders"
OUTDIR = MAINDIR / "data_pipeline" / "mTRF" / "eelbrain"/ "data"
OUTDIR.mkdir(parents=True, exist_ok=True)
LOC_PATH = MAINDIR / "data" / "biosemi_32ch_2mastoid_locs.csv"

EPOCH_DONE = True
N_ORDER = 1   # only needed if you want one specific predictor file
# SUBS = [f"sub-pilot_{i}" for i in range(1, 7)]
SUBS = ['sub-pilot_7']

## Helpers

In [18]:
def find_subject_files(folder: Path, suffix=".mat", contains=None):
    files = sorted(folder.glob(f"*{suffix}"))
    if contains is not None:
        files = [f for f in files if contains in f.name]
    return files

def make_sensor_from_csv(loc_path, chan_labels=None, coords_in_mm=True):
    locs = pd.read_csv(loc_path)

    required_cols = ["ch_name", "x", "y", "z"]
    missing = [c for c in required_cols if c not in locs.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    ch_pos = {
        str(row["ch_name"]): np.array([row["x"], row["y"], row["z"]], dtype=float)
        for _, row in locs.iterrows()
    }

    # MNE expects positions in meters
    if coords_in_mm:
        ch_pos = {k: v / 1000.0 for k, v in ch_pos.items()}

    montage = mne.channels.make_dig_montage(
        ch_pos=ch_pos,
        coord_frame="head",
    )

    if chan_labels is None:
        chan_labels = list(locs["ch_name"].astype(str))

    sensor = Sensor.from_montage(montage, channels=chan_labels)
    return sensor

def parse_block_filename(audio_name: str):
    """
    Assumes names like: Fem_CDS_hap_....wav
    Returns metadata used in the feature dataset.
    """
    stem = Path(audio_name).stem
    stem_clean = stem.replace("_scaled", "")
    parts = stem_clean.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected audio filename format: {audio_name}")

    return {
        "audio_name_original": Path(audio_name).name,
        "audio_stem_clean": stem_clean,
        "gender": parts[0],
        "speech_style": parts[1],
        "emotion": parts[2],
    }

def load_eeglab_mat(mat_path):
    """
    Load a MATLAB v7.3 EEGLAB-style .mat file using mat73.
    Assumes a top-level variable named 'EEG'.
    Returns:
        data    : numpy array, chans x time x epochs
        times   : numpy array, ms
        fs      : float
        reref   : whatever was stored in EEG.ref
        chanlocs: whatever was stored in EEG.chanlocs
    """
    m = mat73.loadmat(str(mat_path))

    if "EEG" not in m:
        raise KeyError(f"'EEG' not found in {mat_path}. Top-level keys: {list(m.keys())}")

    eeg = m["EEG"]

    data = np.asarray(eeg["data"], dtype=float)
    times = np.asarray(eeg["times"], dtype=float).squeeze()
    fs = float(np.asarray(eeg["srate"]).squeeze())

    reref = eeg.get("ref", None)
    chanlocs = eeg.get("chanlocs", None)

    return data, times, fs, reref, chanlocs

def extract_channel_labels(chanlocs, n_channels, sensor_csv_path=None):
    """
    Recover channel labels from EEGLAB chanlocs loaded with mat73.
    """
    def clean_label(x):
        # unwrap one-element list/array
        while isinstance(x, (list, tuple, np.ndarray)) and len(x) == 1:
            x = x[0]
        return str(x).strip()

    labels = []

    if isinstance(chanlocs, list):
        print("channelocs is list")
        for i, ch in enumerate(chanlocs):
            if isinstance(ch, dict) and "labels" in ch and ch["labels"] is not None:
                labels.append(clean_label(ch["labels"]))
            else:
                labels.append(f"ch{i+1:02d}")

    elif isinstance(chanlocs, dict) and "labels" in chanlocs:
        print("channelocs is a dictionary")
        raw = chanlocs["labels"]
        if isinstance(raw, list):
            labels = [clean_label(x) for x in raw]
        else:
            labels = [clean_label(raw)]

    if len(labels) != n_channels:
        if sensor_csv_path is None:
            labels = [f"ch{i+1:02d}" for i in range(n_channels)]
        else:
            locs = pd.read_csv(sensor_csv_path)
            csv_labels = list(locs["ch_name"].astype(str))
            if len(csv_labels) < n_channels:
                raise ValueError(
                    f"Sensor CSV has only {len(csv_labels)} channels, but EEG has {n_channels}."
                )
            labels = csv_labels[:n_channels]

    return labels

def get_audio_duration_ms(audio_file: Path):
    wav = eelbrain.load.wav(audio_file)
    if len(wav.dims) == 2:
        wav = wav.mean("channel")
    return wav.time.tstop * 1000

def trim_epoch(epoch_tc, times_ms, dur_ms):
    start_idx = np.where(times_ms >= 0)[0][0]
    end_idx = np.where(times_ms <= dur_ms)[0][-1]
    return epoch_tc[start_idx:end_idx + 1, :]

In [19]:
def debug_channel_name_mismatch(sensor_csv_path, chan_labels):
    locs = pd.read_csv(sensor_csv_path)
    csv_labels = list(locs["ch_name"].astype(str))

    print("First 10 EEG labels:", chan_labels[:10])
    print("First 10 CSV labels:", csv_labels[:10])

    eeg_only = [ch for ch in chan_labels if ch not in csv_labels]
    csv_only = [ch for ch in csv_labels if ch not in chan_labels]

    print("\nIn EEG but not CSV:", eeg_only)
    print("In CSV but not EEG:", csv_only)

In [30]:
def save_eeg_only_pickle(
    subname,
    datadir=DATA_DIR,
    stimuli_dir=STIMULI_DIR,
    order_dir=ORDER_DIR,
    eeg_outdir=OUTDIR,
    sensor_csv_path=LOC_PATH,
    epoch_done=EPOCH_DONE,
    coords_in_mm=True,
    n_order=N_ORDER,
    ica_mode="ica2",   # "ica", "ica2", or "both"
):
    """
    Save EEG-only Eelbrain Dataset(s) aligned to predefined order in csv.
    Does NOT modify predictor pickle.

    ica_mode:
        "ica"  -> process only *_ica.mat, excluding *_ica_ica2.mat
        "ica2" -> process only *_ica_ica2.mat
        "both" -> process both
    """
    eeg_outdir.mkdir(parents=True, exist_ok=True)

    if epoch_done:
        sub_indir = datadir / subname / "ref_down_filt_chRej" / "epoch_reject" / "ica" / "MAT"
    else:
        sub_indir = datadir / subname / "ref_down_filt_chRej" / "ica" / "epoch_reject" / "MAT"

    all_mat_files = find_subject_files(sub_indir, suffix=".mat", contains="emo")

    if ica_mode == "ica":
        mat_files = [
            f for f in all_mat_files
            if f.stem.endswith("_ica") and not f.stem.endswith("_ica_ica2")
        ]
    elif ica_mode == "ica2":
        mat_files = [
            f for f in all_mat_files
            if f.stem.endswith("_ica_ica2")
        ]
    elif ica_mode == "both":
        mat_files = [
            f for f in all_mat_files
            if f.stem.endswith("_ica") or f.stem.endswith("_ica_ica2")
        ]
    else:
        raise ValueError("ica_mode must be 'ica', 'ica2', or 'both'")

    if not mat_files:
        print(f"No matching {ica_mode} MAT files found for {subname} in {sub_indir}")
        return

    order_file = order_dir / f"order{n_order}.xlsx"
    if not order_file.exists():
        raise FileNotFoundError(f"Missing order file: {order_file}")

    order_df = pd.read_excel(order_file)
    if "audio_path" not in order_df.columns or "stim_idx" not in order_df.columns:
        raise ValueError(f"{order_file} must contain columns 'audio_path' and 'stim_idx'")

    order_df = order_df.loc[
        order_df["stim_idx"].notna() & (order_df["stim_idx"] > 0)
    ].copy()

    order_df["epoch_idx"] = np.arange(len(order_df))
    order_df["filename"] = order_df["audio_path"].map(lambda x: Path(x).name)
    order_df["stim_idx"] = order_df["stim_idx"].astype(int)
    order_df = order_df.sort_values("stim_idx").reset_index(drop=True)

    for mat_file in mat_files:
        print(mat_file)
        fname = mat_file.stem

        if fname.endswith("_ica_ica2"):
            clean_label = "ica2"
        elif fname.endswith("_ica"):
            clean_label = "ica"
        else:
            clean_label = "unknown"

        m = re.search(r"acq-(\d+)", fname)
        if m is None:
            raise ValueError(f"Could not extract acq number from {fname}")
        current_order = int(m.group(1))

        eeg_data, times_ms, fs, reref, chanlocs = load_eeglab_mat(mat_file)
        n_channels = eeg_data.shape[0]

        chan_labels = extract_channel_labels(
            chanlocs,
            n_channels,
            sensor_csv_path=sensor_csv_path
        )

        channel_dim = make_sensor_from_csv(
            sensor_csv_path,
            chan_labels=chan_labels,
            coords_in_mm=coords_in_mm,
        )

        if eeg_data.shape[2] != len(order_df):
            raise ValueError(
                f"{fname}: EEG epochs ({eeg_data.shape[2]}) != valid order rows ({len(order_df)})"
            )

        order_stim = list(order_df["stim_idx"])
        order_name = list(order_df["filename"])

        eeg_trials = []

        for _, row in order_df.iterrows():
            epoch_idx = int(row["epoch_idx"])
            audio_file = stimuli_dir / row["filename"]

            if not audio_file.exists():
                raise FileNotFoundError(f"Missing audio file: {audio_file}")

            dur_ms = get_audio_duration_ms(audio_file)

            epoch_tc = np.asarray(eeg_data[:, :, epoch_idx], dtype=float).T
            epoch_tc = trim_epoch(epoch_tc, times_ms, dur_ms)

            time_dim = UTS(0.0, 1.0 / fs, epoch_tc.shape[0])

            eeg_nd = NDVar(
                epoch_tc,
                dims=(time_dim, channel_dim),
                name="eeg",
                info={"unit": "uV"},
            )

            eeg_trials.append(eeg_nd)

        eeg_ds = Dataset(name=f"{fname}_eeg")
        eeg_ds["stim_idx"] = Var(order_stim, name="stim_idx")
        eeg_ds["stim_name"] = Factor(order_name, name="stim_name")
        eeg_ds["eeg"] = eeg_trials

        eeg_ds.info["preprocessed_eeg"] = fname
        eeg_ds.info["subject"] = subname
        eeg_ds.info["fs"] = fs
        eeg_ds.info["reRef"] = reref
        eeg_ds.info["source_mat"] = str(mat_file)
        eeg_ds.info["sensor_csv"] = str(sensor_csv_path)
        eeg_ds.info["clean_label"] = clean_label
        eeg_ds.info["order"] = current_order

        # out_path = eeg_outdir / f"{subname}_{clean_label}_eeg.pickle"
        out_path = eeg_outdir / f"{subname}_eeg.pickle"
        eelbrain.save.pickle(eeg_ds, out_path)
        print(f"Saved EEG-only dataset: {out_path}")

In [29]:
# run
for sub in SUBS:
    save_eeg_only_pickle(sub)

C:\projects\emo_EEG\data\processed\sub-pilot_7\ref_down_filt_chRej\epoch_reject\ica\MAT\sub-pilot_7_ses-1_task-emo_acq-1_down128Hz_refEXG5_filt1-15Hz_chRej_epoch_reject_ica_ica2.mat
channelocs is list
Saved EEG-only dataset: C:\projects\emo_EEG\data_pipeline\mTRF\eelbrain\data\sub-pilot_7_ica2_eeg.pickle


In [10]:
eeg_ds = eelbrain.load.unpickle(r"C:\projects\emo_EEG\data_pipeline\mTRF\eelbrain\data\sub-pilot_1_eeg.pickle")
print(eeg_ds)
print(eeg_ds.summary())
print(eeg_ds.info['preprocessed_eeg'])

#    stim_idx   stim_name                        eeg            
----------------------------------------------------------------
0    1          Fem_ADS_hap_cont_1_scaled.wav    <NDVar 'eeg'...
1    2          Fem_ADS_hap_cont_2_scaled.wav    <NDVar 'eeg'...
2    3          Fem_ADS_hap_cont_3_scaled.wav    <NDVar 'eeg'...
3    4          Fem_ADS_hap_cont_4_scaled.wav    <NDVar 'eeg'...
4    5          Fem_ADS_sad_cont_1_scaled.wav    <NDVar 'eeg'...
5    6          Fem_ADS_sad_cont_2_scaled.wav    <NDVar 'eeg'...
6    7          Fem_ADS_sad_cont_3_scaled.wav    <NDVar 'eeg'...
7    8          Fem_ADS_sad_cont_4_scaled.wav    <NDVar 'eeg'...
8    9          Fem_CDS_hap_cont_1_scaled.wav    <NDVar 'eeg'...
9    10         Fem_CDS_hap_cont_2_scaled.wav    <NDVar 'eeg'...
10   11         Fem_CDS_hap_cont_3_scaled.wav    <NDVar 'eeg'...
11   12         Fem_CDS_hap_cont_4_scaled.wav    <NDVar 'eeg'...
12   13         Fem_CDS_sad_cont_1_scaled.wav    <NDVar 'eeg'...
13   14         Fem_CDS_s